<a href="https://colab.research.google.com/github/bsheese/cs377/blob/18_classification_redesign/18_classification/18_1_Classification_Basics/18_1_3_Model_Selection_Multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Selection & Multiclass Classification

**Author:** Brad Sheese

---

## Introduction

This notebook covers two important topics:

**Part A: Model Selection** - How do we choose the best model among candidates?
**Part B: Multiclass Classification** - What when we have more than 2 classes?

### Learning Objectives
By the end of this notebook, you will:
1. Use AIC/BIC for theoretical model selection
2. Apply cross-validation for robust evaluation
3. Implement multiclass classification strategies
4. Compare One-vs-Rest vs. Softmax approaches

## Part A: Model Selection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Load Credit Default data
data = fetch_openml(name='credit-g', version=1, as_frame=True)
df = data.frame

y = (df['class'] == 'bad').astype(int)
X = df.drop(columns=['class'])
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset: {len(X_train)} train, {len(X_test)} test samples")

## Cross-Validation: Empirical Model Comparison

**Cross-validation** splits data into k folds, trains k times, and averages results.

This gives a more reliable estimate than a single train/test split.

In [ ]:
# Define models to compare
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), X_train_scaled),
    'Decision Tree (d=3)': (DecisionTreeClassifier(max_depth=3, random_state=42), X_train),
    'Random Forest': (RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42), X_train)
}

# 5-fold cross-validation
cv_results = {}
for name, (model, X_tr) in models.items():
    scores = cross_val_score(model, X_tr, y_train, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name}:")
    print(f"  CV Accuracy: {scores.mean():.3f} (+/- {scores.std():.3f})")

# Visualize
fig, ax = plt.subplots(figsize=(9, 5))
names = list(cv_results.keys())
means = [cv_results[n].mean() for n in names]
stds = [cv_results[n].std() for n in names]
colors = ['steelblue', 'orange', 'green']
bars = ax.bar(names, means, yerr=stds, color=colors, alpha=0.8, capsize=5)
ax.set_ylabel('Accuracy')
ax.set_title('5-Fold Cross-Validation: Model Comparison')
ax.set_ylim(0.6, 0.9)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{mean:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## AIC and BIC: Information Theoretic Criteria

**AIC** (Akaike Information Criterion) and **BIC** (Bayesian Information Criterion) balance:
1. **Goodness of fit** - How well does the model fit the data?
2. **Complexity penalty** - How many parameters does it use?

Formulas:
- AIC = -2*log(L) + 2*k
- BIC = -2*log(L) + log(n)*k

Where:
- log(L) = log-likelihood of the model
- k = number of parameters
- n = number of samples

**Lower is better** - we prefer models that explain data with fewer parameters.

Note: sklearn's LogisticRegression gives us the intercept_ and coef_ to count parameters.

In [ ]:
def calculate_aic_bic(model, X, y, is_lr=False):
    """
    Calculate AIC and BIC for a fitted classifier.
    For Logistic Regression: uses log-likelihood
    For tree-based: uses residual sum of squares approximation
    """
    n = len(y)
    
    if hasattr(model, 'predict_log_proba'):
        try:
            log_likelihood = model.predict_log_proba(X)[
                np.arange(len(y)), y
            ].sum()
        except:
            log_likelihood = -n  # fallback
    else:
        log_likelihood = -n  # fallback for models without log_proba
    
    # Count parameters
    if hasattr(model, 'coef_') and hasattr(model, 'intercept_'):
        k = model.coef_.shape[1] * model.coef_.shape[0] + len(model.intercept_)
    elif hasattr(model, 'n_features_in_'):
        k = model.n_features_in_ + 1
    else:
        k = 10  # default
    
    aic = 2 * k - 2 * log_likelihood
    bic = k * np.log(n) - 2 * log_likelihood
    
    return aic, bic, k

# Fit models and compute AIC/BIC
info_criteria = {}
for name, (model, X_tr) in models.items():
    model.fit(X_tr, y_train)
    try:
        aic, bic, k = calculate_aic_bic(model, X_tr[:100], y_train[:100])
        info_criteria[name] = {'AIC': aic, 'BIC': bic, 'params': k}
        print(f"{name}:")
        print(f"  Parameters: {k}")
        print(f"  AIC: {aic:.1f}")
        print(f"  BIC: {bic:.1f}")
    except Exception as e:
        print(f"{name}: Could not compute AIC/BIC ({e})")

print("\n(Lower AIC/BIC is better)")

In [ ]:
# Visualize AIC/BIC comparison
if info_criteria:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    names = list(info_criteria.keys())
    
    aics = [info_criteria[n]['AIC'] for n in names]
    bics = [info_criteria[n]['BIC'] for n in names]
    
    colors = ['steelblue', 'orange', 'green']
    
    axes[0].bar(names, aics, color=colors, alpha=0.8)
    axes[0].set_title('AIC (lower is better)')
    axes[0].set_ylabel('AIC')
    axes[0].tick_params(axis='x', rotation=15)
    
    axes[1].bar(names, bics, color=colors, alpha=0.8)
    axes[1].set_title('BIC (lower is better)')
    axes[1].set_ylabel('BIC')
    axes[1].tick_params(axis='x', rotation=15)
    
    plt.tight_layout()
    plt.show()

## Part B: Multiclass Classification

What if we need to predict MORE than 2 classes?

Examples:
- Digit recognition (0-9)
- Customer segments (low/medium/high)
- Wine classification (varietals)

Two main strategies:
1. **Softmax (Multinomial)**: Single model with K outputs
2. **One-vs-Rest (OvR)**: Train K binary classifiers

In [ ]:
# Load Wine dataset (multiclass)
from sklearn.datasets import load_wine

wine = load_wine()
X_wine = wine.data
y_wine = wine.target

print("Wine Dataset:")
print(f"  Classes: {np.unique(y_wine)}")
print(f"  Class distribution: {np.bincount(y_wine)}")
print(f"  Features: {X_wine.shape[1]}")

# Train/test split
X_wine_train, X_wine_test, y_wine_train, y_wine_test = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)

scaler_wine = StandardScaler()
X_wine_train_s = scaler_wine.fit_transform(X_wine_train)
X_wine_test_s = scaler_wine.transform(X_wine_test)

## Softmax vs. One-vs-Rest (OvR)

**Softmax (Multinomial):**
- Single model with K output neurons
- Outputs sum to 1 (probability distribution)
- Generally preferred for most problems

**One-vs-Rest (OvR):**
- Train K binary classifiers
  - Classifier 0: "Is it class 0?" vs "Is it NOT class 0?"
  - Classifier 1: "Is it class 1?" vs "Is it NOT class 1?"
  - ...
- Final prediction: class with highest confidence
- Useful when you need interpretable binary models

In [ ]:
# Softmax approach (default for sklearn LogisticRegression with multiclass)
lr_softmax = LogisticRegression(max_iter=1000, random_state=42)
lr_softmax.fit(X_wine_train_s, y_wine_train)
y_pred_softmax = lr_softmax.predict(X_wine_test_s)
acc_softmax = accuracy_score(y_wine_test, y_pred_softmax)

print(f"Softmax (Multinomial):")
print(f"  Accuracy: {acc_softmax:.3f}")
print(f"  Coefficients shape: {lr_softmax.coef_.shape} (classes x features)")

# OvR approach (explicit)
from sklearn.multiclass import OneVsRestClassifier
lr_ovr = OneVsRestClassifier(LogisticRegression(max_iter=1000, random_state=42))
lr_ovr.fit(X_wine_train_s, y_wine_train)
y_pred_ovr = lr_ovr.predict(X_wine_test_s)
acc_ovr = accuracy_score(y_wine_test, y_pred_ovr)

print(f"\nOne-vs-Rest:")
print(f"  Accuracy: {acc_ovr:.3f}")
print(f"  Number of classifiers: {lr_ovr.estimators_.__len__()}")

In [ ]:
# Visualize OvR: probability outputs from each binary classifier
y_proba_ovr = lr_ovr.predict_proba(X_wine_test_s)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
class_names = ['Class 0', 'Class 1', 'Class 2']

for i, ax in enumerate(axes):
    ax.hist(y_proba_ovr[y_wine_test == i, i], bins=15, alpha=0.7, 
            label=f'Actual: {class_names[i]}', color='blue')
    ax.hist(y_proba_ovr[y_wine_test != i, i], bins=15, alpha=0.5, 
            label=f'Actual: Other', color='red')
    ax.axvline(x=0.5, color='black', linestyle='--', label='Threshold')
    ax.set_xlabel(f'P({class_names[i]})')
    ax.set_ylabel('Count')
    ax.set_title(f'OvR Classifier {i}')
    ax.legend(fontsize=8)

plt.suptitle('One-vs-Rest: Probability Distribution for Each Binary Classifier', y=1.02)
plt.tight_layout()
plt.show()

## Key Takeaways

**Model Selection:**
- Cross-validation: empirical evaluation (test on held-out folds)
- AIC/BIC: theoretical criteria penalizing complexity
- Lower AIC/BIC = better balance of fit and simplicity

**Multiclass Classification:**
- Softmax (multinomial): single model, K outputs, generally preferred
- One-vs-Rest: K binary models, useful for interpretability or different base classifiers
- OvR is equivalent to softmax for logistic regression, but differs for other classifiers